<a href="https://colab.research.google.com/github/lawesworks/vision-model-workbench/blob/main/YOLO_OD_Inference_Enablement.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install + Imports

!pip -q install ultralytics opencv-python

import os
import cv2
import numpy as np
from ultralytics import YOLO
from google.colab.patches import cv2_imshow

In [ ]:
# Upload weights

from google.colab import files

# Upload your custom weights (.pt)
uploaded = files.upload()

# If you uploaded multiple files, pick the .pt automatically:
WEIGHTS_PATH = None
for fn in uploaded.keys():
    if fn.endswith(".pt"):
        WEIGHTS_PATH = fn
        break

if WEIGHTS_PATH is None:
    raise FileNotFoundError("No .pt weights uploaded. Please upload your custom .pt file.")

print("Using weights:", WEIGHTS_PATH)

In [ ]:
# Load the Model

model = YOLO(WEIGHTS_PATH)

# Optional: see class names
print("Classes:", model.names)

In [ ]:
# input file (image or video)

uploaded = files.upload()

INPUT_PATH = list(uploaded.keys())[0]
print("Input:", INPUT_PATH)

In [ ]:
# Image Inference and Display Result

def infer_image_cv2(model, image_path, conf=0.25, imgsz=640):
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        raise RuntimeError(f"Failed to read image: {image_path}")

    # Ultralytics expects RGB arrays typically
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    results = model.predict(source=img_rgb, conf=conf, imgsz=imgsz, verbose=False)

    # Ultralytics can render boxes for you
    annotated_rgb = results[0].plot()  # RGB ndarray with drawings

    # Convert back to BGR for cv2_imshow (it’s fine either way, but keep consistent)
    annotated_bgr = cv2.cvtColor(annotated_rgb, cv2.COLOR_RGB2BGR)

    cv2_imshow(annotated_bgr)
    return results

# If INPUT_PATH is an image:
_ = infer_image_cv2(model, INPUT_PATH, conf=0.25, imgsz=640)

In [ ]:
#URL Inputs (image or video)

import urllib.request

def download_url_to_file(url, out_path=None):
    if out_path is None:
        out_path = url.split("?")[0].split("/")[-1]
        if not out_path:
            out_path = "downloaded_file"
    urllib.request.urlretrieve(url, out_path)
    return out_path

# Example:
url = "https://assets2.cbsnewsstatic.com/hub/i/r/2020/02/07/5caefba0-5be0-41a2-8497-b22c1af0783f/thumbnail/1200x630/437169c0c291a67616ed8bf38de2d62a/gettyimages-159406920.jpg"
path = download_url_to_file(url)
infer_image_cv2(model, path)

In [ ]:
import os


INPUT_PATH = "/content/highway.mp4"


#for f in os.listdir("/content"):
#    print(f)

print("Exists:", os.path.exists("/content/highway.mp4"))
print("Size (MB):", os.path.getsize("/content/highway.mp4") / 1024 / 1024)

In [ ]:
# Video inference (OpenCV) + display sampled frames inline
# Colab doesn't handle “live” video windows very well,
# as a workaround, this simple UI:
# - runs inference frame-by-frame,
# - show every Nth frame,
# - and and optionally writes an output video.

def infer_video_cv2(
    model,
    video_path,
    conf=0.25,
    imgsz=640,
    max_frames=None,      # e.g. 200 to limit
    show_every=30,        # show every Nth frame
    save_output=True,
    output_path="output_annotated.mp4"
):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Failed to open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    writer = None
    if save_output:
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        writer = cv2.VideoWriter(output_path, fourcc, fps if fps > 0 else 30, (w, h))

    frame_idx = 0
    last_results = None

    while True:
        ok, frame_bgr = cap.read()
        if not ok:
            break

        frame_idx += 1
        if max_frames is not None and frame_idx > max_frames:
            break

        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        results = model.predict(source=frame_rgb, conf=conf, imgsz=imgsz, verbose=False)
        last_results = results

        annotated_rgb = results[0].plot()
        annotated_bgr = cv2.cvtColor(annotated_rgb, cv2.COLOR_RGB2BGR)

        if writer is not None:
            writer.write(annotated_bgr)

        if show_every and (frame_idx % show_every == 0):
            print(f"Frame {frame_idx}")
            cv2_imshow(annotated_bgr)

    cap.release()
    if writer is not None:
        writer.release()

    if save_output:
        print("Saved:", output_path)

    return last_results

# If INPUT_PATH is a video:
_ = infer_video_cv2(model, INPUT_PATH, conf=0.25, imgsz=640, max_frames=200, show_every=50)

In [ ]:
# Download the Output Video

from google.colab import files
files.download("output_annotated.mp4")

In [ ]:
# RTSP Stream

def infer_rtsp_cv2(model, rtsp_url, conf=0.25, imgsz=640, max_frames=300, show_every=30):
    cap = cv2.VideoCapture(rtsp_url)  # may require ffmpeg / proper backend
    if not cap.isOpened():
        raise RuntimeError("Failed to open RTSP stream. Check URL, network, and backend support.")

    frame_idx = 0
    while True:
        ok, frame_bgr = cap.read()
        if not ok:
            break

        frame_idx += 1
        if max_frames is not None and frame_idx > max_frames:
            break

        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        results = model.predict(source=frame_rgb, conf=conf, imgsz=imgsz, verbose=False)
        annotated_rgb = results[0].plot()
        annotated_bgr = cv2.cvtColor(annotated_rgb, cv2.COLOR_RGB2BGR)

        if show_every and (frame_idx % show_every == 0):
            print(f"RTSP Frame {frame_idx}")
            cv2_imshow(annotated_bgr)

    cap.release()

In [ ]:
# Test RTSP

import cv2

cap = cv2.VideoCapture("rtsp://184.72.239.149/vod/mp4:BigBuckBunny_175k.mov", cv2.CAP_FFMPEG)

if not cap.isOpened():
    print("Failed to open stream")
else:
    print("Stream opened successfully!")

ret, frame = cap.read()
if ret:
    print("Frame shape:", frame.shape)
else:
    print("No frame read")

cap.release()